In [1]:
# !pip install langchain-community

In [2]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

/tmp/ipykernel_891209/986933069.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/dgclasher/Development/projects/ai/langchain_proj/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_pdfs(directory: str) -> list[Document]:
    documents = []

    for pdf_file in Path(directory).glob("*.pdf"):
        docs = PyPDFLoader(str(pdf_file)).load()

        for doc in docs:
            doc.metadata.update({
                "source_file": pdf_file.name,
                "file_type": "pdf"
            })

        documents.extend(docs)

    return documents

docs = load_pdfs("../data/pdf_files")

In [4]:
import torch

if torch.cuda.is_available():
    print("CUDA available")
else:
    print("CUDA not available, using CPU")

CUDA available


In [5]:
## Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [6]:
chunks = split_documents(docs)

Split 108 documents into 284 chunks

Example chunk:
Content: eprints@whiterose.ac.uk
https://eprints.whiterose.ac.uk
Universities of Leeds, Sheffield and York
Deposited via The University of Sheffield.
White Rose Research Online URL for this paper:
https://epri
Metadata: {'producer': 'GPL Ghostscript 9.54.0', 'creator': 'Microsoft® Word 2013', 'creationdate': "D:20260208191936Z00'00'", 'moddate': "D:20260208191936Z00'00'", 'author': 'Wells, V.K.', 'title': 'The influence of behavioural psychology on consumer psychology and marketing', 'keywords': 'behaviourism; behavioural psychology; operant conditioning; classical conditioning; consumer behaviour analysis; foraging', 'source': '../data/pdf_files/influence.pdf', 'total_pages': 57, 'page': 0, 'page_label': '1', 'source_file': 'influence.pdf', 'file_type': 'pdf'}


### Embedding and Vector store DB

In [8]:
import os
from huggingface_hub import login

HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    login(HF_TOKEN)

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

## Embedding class
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.device = "cpu"
        self.model = None
        self._load_model()

    def _load_model(self):
        preferred_device = "cpu"
        try:
            print(f"Loading embedding model: {self.model_name} on {preferred_device}")
            self.model = SentenceTransformer(self.model_name, device=preferred_device)
            self.device = preferred_device
            print(f"Model loaded successfully on {self.device}, embedding dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"CUDA setup failed for {self.model_name}: {e}")
            print("Falling back to CPU because CUDA is not usable in this environment.")
            self.device = "cpu"
            self.model = SentenceTransformer(self.model_name, device="cpu")
            print(f"Model loaded successfully on {self.device}, embedding dimensions: {self.model.get_embedding_dimension()}")

    def generate_embedding(self, text: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embedding for {len(text)} texts on {self.device}...")
        embeddings = self.model.encode(text, show_progress_bar=True, convert_to_numpy=True)
        print(f"Generated embedding with shape: {embeddings.shape}")
        return embeddings

## Initialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2 on cpu


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3667.59it/s]


Model loaded successfully on cpu, embedding dimensions: 384


In [14]:
### VectorStore

import os

class VectorStore:
  def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./vector_store"):
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.client = None
    self.collection = None
    self._initialize_store()

  def _initialize_store(self):
    try:
      os.makedirs(self.persist_directory, exist_ok=True)
      self.client = chromadb.PersistentClient(path=self.persist_directory)

      self.collection = self.client.get_or_create_collection(
          name=self.collection_name,
          metadata={"description": "PDF document embeddings for RAG"}
      )
      print(f"Vector datastore initialized, Collection: {self.collection_name}")
      print(f"Existing documents in collection: {self.collection.count()}")
    except Exception as e:
      print(f"Error initializing the vector store: {e}")
      raise

  def add_documents(self, documents: List[Any], embeddings: np.ndarray):
    if len(documents) != len(embeddings):
      raise ValueError("Number of documents and embeddings must be the same")

    print(f"Adding {len(documents)} documents to the vector store...")

    ids = []
    metadatas = []
    documents_text = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
      doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
      ids.append(doc_id)

      metadata = dict(doc.metadata)
      metadata['doc_index'] = i
      metadata['content_length'] = len(doc.page_content)
      metadatas.append(metadata)

      documents_text.append(doc.page_content)
      embeddings_list.append(embedding.tolist())

    try:
      self.collection.add(
          ids=ids,
          documents=documents_text,
          metadatas=metadatas,
          embeddings=embeddings_list
      )
    except Exception as e:
      print(f"Error adding document to the vector store: {e}")
      raise

    print(f"Added {len(documents)} documents to the vector store")
    print(f"Total documents in collection: {self.collection.count()}")

vectorstore = VectorStore()
vectorstore


Vector datastore initialized, Collection: pdf_documents
Existing documents in collection: 0


In [15]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embedding(texts)
vectorstore.add_documents(chunks, embeddings)

Generating embedding for 284 texts on cpu...


Batches: 100%|██████████| 9/9 [00:11<00:00,  1.29s/it]


Generated embedding with shape: (284, 384)
Adding 284 documents to the vector store...
Added 284 documents to the vector store
Total documents in collection: 284


In [16]:
class RAGRetriever:
  def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
    self.vector_store = vector_store
    self.embedding_manager = embedding_manager

  def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
    """
    Retrieves the top_k most relevant documents from the vector store based on a query.
    It uses the provided EmbeddingManager to generate the query embedding.

    Args:
      query (str): The search query.
      top_k (int): The number of top relevant documents to retrieve.

    Returns:
      List[Dict[str, Any]]: A list of dictionaries, where each dictionary represents
                            a retrieved document with its content, metadata, ChromaDB's distance,
                            and the calculated cosine similarity score.
    """
    if not query:
      raise ValueError("Query cannot be empty.")

    print(f"Generating embedding for query: '{query}' using EmbeddingManager...")
    # Generate embedding for the query. Keep it as a numpy array for cosine similarity calculation.
    query_embedding_np = self.embedding_manager.generate_embedding([query])[0] # This is a numpy array

    print(f"Retrieving top {top_k} documents from the vector store collection: {self.vector_store.collection_name}...")
    # Query the ChromaDB collection, including document embeddings
    results = self.vector_store.collection.query(
        query_embeddings=[query_embedding_np.tolist()], # ChromaDB expects list of lists for query_embeddings
        n_results=top_k,
        include=['documents', 'metadatas', 'distances', 'embeddings'] # Include actual embeddings
    )

    retrieved_documents = []
    if results and results['documents'] and results['documents'][0]:
      print(f"Found {len(results['documents'][0])} documents.")
      for i in range(len(results['documents'][0])):
        doc_content = results['documents'][0][i]
        doc_metadata = results['metadatas'][0][i]
        doc_distance = results['distances'][0][i] # This is typically L2 distance from ChromaDB

        # Get the retrieved document embedding (as a list from Chroma, convert to numpy array)
        retrieved_doc_embedding_list = results['embeddings'][0][i]
        retrieved_doc_embedding_np = np.array(retrieved_doc_embedding_list)

        # Calculate cosine similarity between query and retrieved document embeddings
        # Reshape for sklearn's cosine_similarity to work with single vectors
        cosine_sim_score = cosine_similarity(
            query_embedding_np.reshape(1, -1),
            retrieved_doc_embedding_np.reshape(1, -1)
        )[0][0] # cosine_similarity returns a 2D array, get the single score

        retrieved_documents.append({
            "content": doc_content,
            "metadata": doc_metadata,
            "chroma_distance": doc_distance, # Renamed for clarity: this is Chroma's L2 distance
            "cosine_similarity_score": cosine_sim_score # New: actual cosine similarity
        })
    else:
      print("No documents found for the given query.")

    return retrieved_documents

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [17]:
rag_retriever.retrieve("Jews")

Generating embedding for query: 'Jews' using EmbeddingManager...
Generating embedding for 1 texts on cpu...


Batches: 100%|██████████| 1/1 [00:00<00:00, 90.74it/s]

Generated embedding with shape: (1, 384)
Retrieving top 5 documents from the vector store collection: pdf_documents...
Found 5 documents.


[{'content': 'I know much about this. I just know it was a poor social structure that was potentially dangerous, \nand that is why it had to change. It was unjust and irrational. \nParticipant 24: Nothing. \nParticipant 25: I know that Germans believed that Jews were filthy and that they were the reason \nfor something like the economy dropping or something? I also feel like they hated Jews because \nof something having to do with Jesus and how the Jews "killed" Jesus in a way, thinking he \nwasn\'t their messiah? I\'m not sure, though. I do know that the Germans were brainwashed by \nHitler and that even though lots of Germans were actually Jews, that wasn\'t good enough and \nthey wanted to get rid of Jews all together. \nParticipant 26: I know fascism is the idea that everyone in to obey a strict authoritarian \ngovernment, and that medical science had focused on eugenics at the time. There was a lot of \npatriotism and  no one was allowed to oppose the political agenda.',
  'metada

### Simple RAG pipeline with Groq

In [30]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model_name="qwen/qwen3.8-27b", max_tokens=2048)
# llm.invoke("What is the capital of France? give me a short answer")

In [34]:
## Simple RAG function
def simple_rag(query, retriever, llm, top_k=5):
    """
    A simple RAG function that retrieves relevant documents based on a query
    and generates a response using the provided LLM.

    Args:
        query (str): The input query for retrieval.
        retriever (RAGRetriever): An instance of RAGRetriever to fetch relevant documents.
        llm (ChatGroq): An instance of the LLM to generate responses.
        top_k (int): The number of top relevant documents to retrieve.

    Returns:
        str: The generated response from the LLM based on the retrieved documents.
    """
    # Retrieve relevant documents
    retrieved_docs = retriever.retrieve(query, top_k=top_k)

    # Prepare context for the LLM
    context = "\n\n".join([f"Document {i+1}:\n{doc['content']}" for i, doc in enumerate(retrieved_docs)])

    if not context:
        return "No relevant documents found for the given query."

    # Generate response using the LLM
    prompt = f"Based on the following documents, answer the query: '{query}'\n\n{context}"
    response = llm.invoke(prompt)

    return response.content

In [35]:
query = "Tell me about the history of Jews in Europe."
response = simple_rag(query, rag_retriever, llm, top_k=5)
print(response)

Generating embedding for query: 'Tell me about the history of Jews in Europe.' using EmbeddingManager...
Generating embedding for 1 texts on cpu...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.83it/s]


Generated embedding with shape: (1, 384)
Retrieving top 5 documents from the vector store collection: pdf_documents...
Found 5 documents.
Based on the provided documents, the history of Jews in Europe is primarily described through the lens of 20th-century German fascism and the Holocaust, rather than a broad historical overview. The key points derived from the texts are:

*   **Antisemitic Propaganda and Persecution:** German fascists and Nazi leaders, particularly Adolf Hitler, promoted antisemitic propaganda that portrayed Jews as "parasites" responsible for economic failures. There were also references to historical religious animosity, such as the belief that Jews "killed" Jesus.
*   **The Holocaust:** Under Hitler’s leadership, the Nazi regime systematically targeted Jews for genocide. Millions of Jews were killed in concentration camps or murdered. The regime enforced strict policies, punishing non-Jews who hid Jews.
*   **Broader Context of Oppression:** The persecution of Jews

### Enhanced RAG pipeline

In [40]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    An advanced RAG function that retrieves relevant documents based on a query,
    filters them based on a minimum cosine similarity score, and generates a response
    using the provided LLM.

    Args:
        query (str): The input query for retrieval.
        retriever (RAGRetriever): An instance of RAGRetriever to fetch relevant documents.
        llm (ChatGroq): An instance of the LLM to generate responses.
        top_k (int): The number of top relevant documents to retrieve.
        min_score (float): The minimum cosine similarity score for filtering documents.
        return_context (bool): Whether to return the context used for the LLM.

    Returns:
        str: The generated response from the LLM based on the retrieved documents.
        Optional[str]: The context used for the LLM if return_context is True.
    """
    results = retriever.retrieve(query, top_k=top_k)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', 'unknown'),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['cosine_similarity_score'],
        'preview': doc['content'][:100]  # Optional: Include a preview of the content
    } for doc in results]

    confidence = max(doc['cosine_similarity_score'] for doc in results)

    prompt = f"Based on the following documents, answer the query: '{query}'\n\n{context}"
    response = llm.invoke([prompt.format(query=query, context=context)])
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

In [42]:
query = "What is the history of Jews in Europe?"
rag_response = rag_advanced(query, rag_retriever, llm, top_k=5, min_score=0.2, return_context=True)
print("Answer: ", rag_response['answer'])
print("Sources: ", rag_response['sources'])
print("Confidence: ", rag_response['confidence'])
print("Preview of context: ", rag_response.get('context', '')[:200])  # Print first 500 chars of context

Generating embedding for query: 'What is the history of Jews in Europe?' using EmbeddingManager...
Generating embedding for 1 texts on cpu...


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.49it/s]

Generated embedding with shape: (1, 384)
Retrieving top 5 documents from the vector store collection: pdf_documents...
Found 5 documents.


Answer:  Based on the provided documents, there is no general history of Jews in Europe. The text exclusively contains limited, fragmented, and often inaccurate participant accounts focused on German fascism, Hitler’s anti-Semitic ideology, and the Holocaust.
Sources:  [{'source': 'psychoanalytical.pdf', 'page': 49, 'score': np.float64(0.4936659061090572), 'preview': 'I know much about this. I just know it was a poor social structure that was potentially dangerous, \n'}, {'source': 'psychoanalytical.pdf', 'page': 46, 'score': np.float64(0.4551229481452945), 'preview': 'the economic failure and also viewed them as "parasites" for that reason, and that is a primary \nrea'}, {'source': 'psychoanalytical.pdf', 'page': 45, 'score': np.float64(0.43853415579900457), 'preview': 'Participant 13: Became the leader of the Nazi Party following WWI. Directly responsible for the \nHol'}, {'source': 'psychoanalytical.pdf', 'page': 45, 'score': np.float64(0.41900567530284333), 'preview': 'If you were 